# IdeaWeaver SLM Builder

Configure a from-scratch, Gemma-4-Nano-style small language model — interleaved local/global attention, grouped-query attention, QK-norm, partial RoPE, and cross-layer KV-cache sharing — and see live parameter and VRAM estimates as you go.

**To run this, press "Runtime" → "Run all" on a free Tesla T4 Google Colab instance!** The GPU isn't required for this notebook (the builder itself is a static UI), but launching it the same way you'd launch a real training job keeps this notebook reusable once GPU-backed training lands.

[GitHub](https://github.com/ideaweaver-ai/ideaweaver-slm-builder) · [IdeaWeaver AI Labs](https://www.ideaweaver.ai) · [Building Small Language Models from Scratch (course)](https://www.ideaweaver.ai/courses)

> This notebook clones the repo, installs Node.js + dependencies, starts the app, and displays it below as an embedded iframe — the same pattern Unsloth Studio uses. **This build is a UI/config preview**: "Start Training" in the app simulates a loss curve so you can see the panel working; it does not train a real model in this Colab session yet.

### Setup: clone the repo

In [ ]:
!git clone --depth 1 --branch main https://github.com/ideaweaver-ai/ideaweaver-slm-builder.git
%cd ideaweaver-slm-builder

### Install Node.js 20 and project dependencies

Colab's base image doesn't ship a recent-enough Node.js for Next.js 16, so this installs Node 20 from NodeSource first.

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y nodejs > /dev/null 2>&1
!node -v && npm -v
!npm install --no-audit --no-fund

### Start IdeaWeaver SLM Builder

Starts the dev server in the background, waits for it to be ready, then displays it in an embedded iframe below — the same `serve_kernel_port_as_iframe` trick Unsloth Studio uses to run a local web app inside a Colab cell.

In [ ]:
import subprocess, threading, time

PORT = 3000

proc = subprocess.Popen(
    ["npm", "run", "dev", "--", "-p", str(PORT)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

ready = threading.Event()

def _pump():
    for line in proc.stdout:
        print(line, end="")
        if "Ready in" in line:
            ready.set()

threading.Thread(target=_pump, daemon=True).start()

if not ready.wait(timeout=60):
    raise RuntimeError("Server didn't start in time — check the log output above.")

time.sleep(1)  # small buffer so the first request doesn't race the server

from google.colab import output
output.serve_kernel_port_as_iframe(PORT, height=1200, width="100%")

---

Questions or bugs? Open an issue on [GitHub](https://github.com/ideaweaver-ai/ideaweaver-slm-builder/issues). Want to understand every one of these variables — attention internals, RoPE, KV caching, and the training loop — not just tune them? Check out [Building Small Language Models from Scratch](https://www.ideaweaver.ai/courses).